In [0]:
from pyspark.sql import functions as F

CONTROL_TABLE = (
    "target_catalog.control.ingestion_control"
)

configs = (
    spark.table(CONTROL_TABLE)
    .select(
        "table_id",
        "target_table",
        "primary_key"
    )
    .collect()
)

for config in configs:

    table_id = config["table_id"]
    target_table = config["target_table"]
    primary_key = config["primary_key"]

    target_df = spark.table(target_table)

    total_rows = target_df.count()

    duplicate_keys = (
        target_df
        .groupBy(primary_key)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    null_keys = (
        target_df
        .filter(F.col(primary_key).isNull())
        .count()
    )

    print(f"Table: {table_id}")
    print(f"Target records: {total_rows}")
    print(f"Duplicate primary keys: {duplicate_keys}")
    print(f"Null primary keys: {null_keys}")
    print("-" * 40)


display(
    spark.table(CONTROL_TABLE).select(
        "table_id",
        "last_processed_value",
        "rows_processed",
        "status",
        "last_run_at",
        "error_message"
    ).orderBy("table_id")
)